In [0]:
# Databricks notebook source

from pyspark.sql import functions as F


# ============================================================
# CONFIGURATION
# ============================================================

SOURCE_PATH = "/Volumes/workspace/default/fiar-docs/"

SCHEMA_PATH = (
    "/Volumes/workspace/default/fiar-docs/_schema/document_repository_binary"
)

CHECKPOINT_PATH = (
    "/Volumes/workspace/default/fiar-docs/_checkpoints/document_repository_binary"
)

TARGET_TABLE = "discovery_document_repository"


# ============================================================
# 1. AUTO LOADER
#
# binaryFile is used because we need FILE METADATA:
#
#   path
#   modificationTime
#   length
#   content
#
# We are NOT parsing the Markdown document.
# ============================================================

df_files = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.includeExistingFiles", "true")
        .option("recursiveFileLookup", "true")
        .load(SOURCE_PATH)
)

df_files.printSchema()


# ============================================================
# 2. BUILD DOCUMENT METADATA
# ============================================================

df_metadata = (
    df_files

    # --------------------------------------------------------
    # Only Markdown documents
    # --------------------------------------------------------

    .filter(
        F.lower(F.col("path")).endswith(".md")
    )

    # Ignore Auto Loader internal folders
    .filter(
        ~F.col("path").contains("/_schema/")
    )
    .filter(
        ~F.col("path").contains("/_checkpoints/")
    )

    # --------------------------------------------------------
    # STORAGE URI
    # --------------------------------------------------------

    .withColumn(
        "storage_uri",
        F.col("path")
    )

    # --------------------------------------------------------
    # FILE NAME
    # --------------------------------------------------------

    .withColumn(
        "file_name",
        F.element_at(
            F.split(
                F.col("path"),
                "/"
            ),
            -1
        )
    )

    # --------------------------------------------------------
    # DOCUMENT TYPE
    #
    # Example:
    #
    # /Volumes/workspace/default/fiar-docs/invoice/INV-123.md
    #
    #                         ↓
    #                      invoice
    # --------------------------------------------------------

    .withColumn(
        "document_type",
        F.regexp_extract(
            F.col("path"),
            r"/fiar-docs/([^/]+)/[^/]+$",
            1
        )
    )

    # --------------------------------------------------------
    # SOURCE FORMAT
    # --------------------------------------------------------

    .withColumn(
        "source_format",
        F.lit("md")
    )

    # --------------------------------------------------------
    # CONTENT HASH
    #
    # binaryFile.content is binary.
    # SHA2 can hash the binary content.
    # --------------------------------------------------------

    .withColumn(
        "content_hash",
        F.sha2(
            F.col("content"),
            256
        )
    )

    # --------------------------------------------------------
    # DOCUMENT ID
    #
    # Based on storage URI.
    #
    # Filename/business keys are NOT used.
    # --------------------------------------------------------

    .withColumn(
        "document_id",
        F.abs(
            F.xxhash64(
                F.col("storage_uri")
            )
        )
    )

    # --------------------------------------------------------
    # SOURCE SYSTEM
    # --------------------------------------------------------

    .withColumn(
        "source_system",
        F.lit("FIAR")
    )

    # --------------------------------------------------------
    # INGESTION TIME
    # --------------------------------------------------------

    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )

    # --------------------------------------------------------
    # GOVERNANCE METADATA
    #
    # Not available from basic file metadata.
    # Leave NULL until the governance process populates them.
    # --------------------------------------------------------

    .withColumn(
        "reporting_period",
        F.lit(None).cast("string")
    )

    .withColumn(
        "cui_marked",
        F.lit(None).cast("boolean")
    )

    .withColumn(
        "contains_pii",
        F.lit(None).cast("boolean")
    )

    # --------------------------------------------------------
    # AUTO LOADER FILE MODIFICATION TIME
    #
    # Used only for ordering if required.
    # --------------------------------------------------------

    .withColumn(
        "__sequence",
        F.col("modificationTime")
    )

    .select(
        "document_id",
        "document_type",
        "file_name",
        "storage_uri",
        "content_hash",
        "ingested_at",
        "source_format",
        "source_system",
        "reporting_period",
        "cui_marked",
        "contains_pii",
        "__sequence"
    )
)


# ============================================================
# 3. WRITE NEW DOCUMENT METADATA
#
# NO MERGE INTO
#
# Existing documents are identified by storage_uri.
# ============================================================

def process_documents(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    # Remove duplicate files in the same batch
    batch_df = (
        batch_df
        .dropDuplicates(["storage_uri"])
    )

    # Existing repository records
    existing = (
        spark.table(TARGET_TABLE)
        .select("storage_uri")
        .distinct()
    )

    # Keep only files not already registered
    new_documents = (
        batch_df
        .join(
            existing,
            on="storage_uri",
            how="left_anti"
        )
    )

    # Remove internal CDC column before writing
    new_documents = (
        new_documents
        .drop("__sequence")
    )

    # Append metadata
    (
        new_documents
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )


# ============================================================
# 4. UPDATE CONSTRAINT TO ALLOW 'md' FORMAT
# ============================================================

spark.sql(f"ALTER TABLE {TARGET_TABLE} DROP CONSTRAINT IF EXISTS chk_dr_format")
spark.sql(
    f"ALTER TABLE {TARGET_TABLE} ADD CONSTRAINT chk_dr_format "
    f"CHECK (source_format IS NULL OR source_format IN "
    f"('pdf', 'csv', 'xls', 'xlsx', 'dbf', 'fixed_width', 'txt', 'md'))"
)

# ============================================================
# 5. START AUTO LOADER
# ============================================================

query = (
    df_metadata
        .writeStream
        .foreachBatch(process_documents)
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .trigger(
            availableNow=True
        )
        .start()
)

query.awaitTermination()

In [0]:
%sql
select * from discovery_document_repository
